# Plot regions

In [31]:
import numpy as np
import pyvista as pv
import geovista as gv
from geovista.common import to_cartesian, RADIUS
from shapely.geometry import box
from shapely.ops import unary_union
import ipywidgets as widgets
from IPython.display import display

In [33]:
region_bounds = {
    1: [79.921875, 89.953125, -179.92969, 179.92969],
    2: [24.984375, 80.015625, -45.070312, 45.070312],
    3: [24.984375, 80.015625, 44.929688, 135.07031],
    4: [24.984375, 80.015625, -179.92969, 179.92969],
    5: [24.984375, 80.015625, -135.07031, -44.929688],
    6: [-25.078125, 25.078125, -45.070312, 45.070312],
    7: [-25.078125, 25.078125, 44.929688, 135.07031],
    8: [-25.078125, 25.078125, -179.92969, 179.92969],
    9: [-25.078125, 25.078125, -135.07031, -44.929688],
    10: [-80.015625, -24.984375, -45.070312, 45.070312],
    11: [-80.015625, -24.984375, 44.929688, 135.07031],
    12: [-80.015625, -24.984375, -179.92969, 179.92969],
    13: [-80.015625, -24.984375, -135.07031, -44.929688],
    14: [-89.953125, -79.921875, -179.92969, 179.92969],
}

domains = {
    "SA": [6, 9, 10, 13],
    "NA": [2, 3, 6, 7],
    "CHINA": [3, 4, 7, 8],
    "INDIA": [3, 7],
}

colours = {
    "SA": "red",
    "NA": "orange",
    "CHINA": "cyan",
    "INDIA": "magenta",
}

In [34]:
def tile_outline(lat_min, lat_max, lon_min, lon_max, n=120, radius=RADIUS):
    lons_top = np.linspace(lon_min, lon_max, n)
    lats_top = np.full(n, lat_max)

    lats_right = np.linspace(lat_max, lat_min, n)
    lons_right = np.full(n, lon_max)

    lons_bot = np.linspace(lon_max, lon_min, n)
    lats_bot = np.full(n, lat_min)

    lats_left = np.linspace(lat_min, lat_max, n)
    lons_left = np.full(n, lon_min)

    lons = np.concatenate([lons_top, lons_right, lons_bot, lons_left])
    lats = np.concatenate([lats_top, lats_right, lats_bot, lats_left])

    xyz = to_cartesian(lons, lats, radius=radius)
    npts = xyz.shape[0]
    lines = np.hstack([[npts + 1], np.arange(npts), 0])
    return pv.PolyData(xyz, lines=lines)

def domain_union_outline(domain_codes, region_bounds, radius=RADIUS):
    polys = []
    for code in domain_codes:
        lat_min, lat_max, lon_min, lon_max = region_bounds[code]
        polys.append(box(lon_min, lat_min, lon_max, lat_max))

    union = unary_union(polys)
    geoms = [union] if union.geom_type == "Polygon" else list(union.geoms)

    outlines = []
    for poly in geoms:
        lonlat = np.asarray(poly.exterior.coords)
        xyz = to_cartesian(lonlat[:, 0], lonlat[:, 1], radius=radius)
        npts = xyz.shape[0]
        lines = np.hstack([[npts], np.arange(npts)])
        outlines.append(pv.PolyData(xyz, lines=lines))

    merged = outlines[0]
    for o in outlines[1:]:
        merged = merged.merge(o)
    return merged.clean()

def domain_centroid_lonlat(domain_codes, region_bounds):
    lons_mid, lats_mid, weights = [], [], []
    for code in domain_codes:
        lat_min, lat_max, lon_min, lon_max = region_bounds[code]
        lat_c = 0.5 * (lat_min + lat_max)
        lon_c = 0.5 * (lon_min + lon_max)
        area = (lat_max - lat_min) * (lon_max - lon_min)
        w = max(area, 1e-9) * np.cos(np.deg2rad(lat_c))
        lons_mid.append(lon_c)
        lats_mid.append(lat_c)
        weights.append(w)

    return np.average(lons_mid, weights=weights), np.average(lats_mid, weights=weights)


In [36]:
p = gv.GeoPlotter()
p.add_base_layer(texture=gv.natural_earth_hypsometric())
p.add_coastlines()
p.add_graticule()

domain_actors = {}
tile_actors = {}
label_actors = {}

for name, codes in domains.items():
    tile_actors[name] = []

    # draw each tile
    for code in codes:
        lat_min, lat_max, lon_min, lon_max = region_bounds[code]
        rect = tile_outline(lat_min, lat_max, lon_min, lon_max)
        a = p.add_mesh(rect, color=colours[name], line_width=1)
        tile_actors[name].append(a)

        # Optional filled tile
        filled = rect.delaunay_2d()
        af = p.add_mesh(filled, color=colours[name], opacity=0.08)
        tile_actors[name].append(af)

    # merged domain border
    outline = domain_union_outline(codes, region_bounds)
    domain_actors[name] = p.add_mesh(outline, color=colours[name], line_width=4)

    # label
    lon_c, lat_c = domain_centroid_lonlat(codes, region_bounds)
    xyz = to_cartesian(np.array([lon_c]), np.array([lat_c]), radius=RADIUS)
    label_actors[name] = p.add_point_labels(
        xyz, [name], font_size=18, point_size=8, shape=None, always_visible=True
    )

p.view_xy()
p.show()

# -----------------------------
# INTERACTIVE TOGGLES
# -----------------------------

def set_vis(obj, visible):
    if hasattr(obj, "SetVisibility"):
        obj.SetVisibility(1 if visible else 0)

def toggle_domain(name, visible):
    set_vis(domain_actors[name], visible)
    for a in tile_actors[name]:
        set_vis(a, visible)
    try:
        set_vis(label_actors[name], visible)
    except:
        pass
    p.render()

controls = []
for name in domains:
    cb = widgets.Checkbox(value=True, description=name)
    cb.observe(lambda ch, nm=name: toggle_domain(nm, ch["new"]), names="value")
    controls.append(cb)

display(widgets.HBox(controls))

/var/folders/h6/4k44gb556kj8qqy4tznsdtmr0000gq/T/ipykernel_34660/2319622008.py:17: UserWarning: geovista found no coordinate reference system (CRS) attached to mesh.
  a = p.add_mesh(rect, color=colours[name], line_width=1)
/var/folders/h6/4k44gb556kj8qqy4tznsdtmr0000gq/T/ipykernel_34660/2319622008.py:22: UserWarning: geovista found no coordinate reference system (CRS) attached to mesh.
  af = p.add_mesh(filled, color=colours[name], opacity=0.08)
/var/folders/h6/4k44gb556kj8qqy4tznsdtmr0000gq/T/ipykernel_34660/2319622008.py:27: UserWarning: geovista found no coordinate reference system (CRS) attached to mesh.
  domain_actors[name] = p.add_mesh(outline, color=colours[name], line_width=4)
/opt/anaconda3/envs/geovista/lib/python3.11/site-packages/pyvista/plotting/plotter.py:5705: UserWarning: geovista found no coordinate reference system (CRS) attached to mesh.
  self.add_mesh(


Widget(value='<iframe src="http://localhost:64143/index.html?ui=P_0x20cee7e90_15&reconnect=auto" class="pyvist…

# WAVEWATCH III

In [5]:
import geovista as gv
from geovista.pantry.data import ww3_global_tri
import geovista.theme
import pyvista as pv
#pv.set_jupyter_backend("trame")


In [6]:

# Load the sample data.
sample = ww3_global_tri()

# Create the mesh from the sample data.
mesh = gv.Transform.from_unstructured(
    sample.lons, sample.lats, connectivity=sample.connectivity, data=sample.data
)

In [7]:
# Plot the mesh.
p = gv.GeoPlotter()
sargs = {"title": f"{sample.name} / {sample.units}"}
p.add_mesh(mesh, show_edges=True, scalar_bar_args=sargs)
p.add_base_layer(texture=gv.natural_earth_hypsometric())
p.add_coastlines()
p.add_graticule()
p.view_xy(negative=True)
p.add_axes()
p.show()

Widget(value='<iframe src="http://localhost:64143/index.html?ui=P_0x1a17bd090_1&reconnect=auto" class="pyvista…

In [15]:
import geovista as gv
from geovista.pantry.data import nemo_orca2
import geovista.theme

# Load sample data.
sample = nemo_orca2()

# Create the mesh from the sample data.
mesh = gv.Transform.from_2d(sample.lons, sample.lats, data=sample.data)

# Remove cells from the mesh with NaN values.
mesh = mesh.threshold()

# Plot the mesh.
p = gv.GeoPlotter()
sargs = {"title": f"{sample.name} / {sample.units}"}
p.add_mesh(mesh, show_edges=True, scalar_bar_args=sargs)
p.add_base_layer(texture=gv.natural_earth_1())
p.add_coastlines()
p.view_xy()
p.add_axes()
p.show()

Widget(value='<iframe src="http://localhost:64143/index.html?ui=P_0x1c8b6fe90_6&reconnect=auto" class="pyvista…